# TabPFN Standalone Experiment (Stage 1 & Stage 2)

A lightweight, **100% standalone** notebook to test `TabPFN` on our nested `1,000 Train → 500 Test` subsets without re-running `xai_modeling.ipynb`.

- Reads `TABPFN_TOKEN` automatically from `.env` (gitignored).
- Tries **TabPFN v3.5** first; if Prior Labs license/terms are unavailable, falls back cleanly to ungated **TabPFN v2** (`ModelVersion.V2`).
- Saves predictions to `artifacts/stage{1,2}_tabpfn_preds.npz`.

In [1]:
# Stage 1 TabPFN Experiment (1,000 Train -> 500 Test, standalone & self-contained)
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

# Load .env cleanly without hardcoding secrets in the notebook
_env_file = Path(".env")
if _env_file.exists():
    for _line in _env_file.read_text().splitlines():
        _line = _line.strip()
        if _line and not _line.startswith("#") and "=" in _line:
            _k, _v = _line.split("=", 1)
            os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))

print(f"[Stage 1] TABPFN_TOKEN set in env: {bool(os.environ.get('TABPFN_TOKEN'))}", flush=True)

t0 = time.perf_counter()
df_tr = pd.read_csv(Path("processed") / "stage1_tfm_train.csv")
df_te = pd.read_csv(Path("processed") / "stage1_tfm_test.csv")

tr_sub, _ = train_test_split(df_tr, train_size=1000, stratify=df_tr["accepted"], random_state=42)
te_sub, _ = train_test_split(df_te, train_size=500, stratify=df_te["accepted"], random_state=42)

_num_cols = ["loan_amnt", "dti", "risk_score", "emp_length_years"]
_cat_cols = ["state"]
_feats = _num_cols + _cat_cols

tfm_prep = ColumnTransformer(
    [
        ("num", SimpleImputer(strategy="median"), _num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ]), _cat_cols),
    ],
    remainder="drop",
)
X_tfm_tr = tfm_prep.fit_transform(tr_sub[_feats])
X_tfm_te = tfm_prep.transform(te_sub[_feats])
print(f"[Stage 1] Data ready: train={X_tfm_tr.shape}, test={X_tfm_te.shape}", flush=True)

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

try:
    print(f"[Stage 1] Trying TabPFN v3.5 (n_estimators=4, device='auto')...", flush=True)
    clf = TabPFNClassifier(n_estimators=4, device="auto", show_progress_bar=True, random_state=42)
    clf.fit(X_tfm_tr, tr_sub["accepted"])
    ver_used = "v3.5"
except Exception as e:
    print(f"[Stage 1] v3.5 unavailable ({type(e).__name__}: {e}); falling back to ungated TabPFN v2...", flush=True)
    clf = TabPFNClassifier.create_default_for_version(
        ModelVersion.V2, n_estimators=4, device="auto", show_progress_bar=True, random_state=42
    )
    clf.fit(X_tfm_tr, tr_sub["accepted"])
    ver_used = "v2"

print(f"[Stage 1 ({ver_used})] Fit done in {time.perf_counter() - t0:.1f}s. Predicting...", flush=True)
p_tfm = clf.predict_proba(X_tfm_te)[:, 1]
Path("artifacts").mkdir(exist_ok=True)
np.savez_compressed("artifacts/stage1_tabpfn_preds.npz", y_true=te_sub["accepted"].to_numpy(), y_prob=p_tfm)
print(
    f"[Stage 1 ({ver_used})] Done in {time.perf_counter() - t0:.1f}s | "
    f"Test ROC-AUC: {roc_auc_score(te_sub['accepted'], p_tfm):.4f} | "
    f"Test PR-AUC: {average_precision_score(te_sub['accepted'], p_tfm):.4f}",
    flush=True,
)

[Stage 1] TABPFN_TOKEN set in env: True
[Stage 1] Data ready: train=(1000, 5), test=(500, 5)
[Stage 1] Trying TabPFN v3.5 (n_estimators=4, device='auto')...


tabpfn-v3.5-20260909.safetensors:   0%|          | 0.00/876M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

[Stage 1 (v3.5)] Fit done in 27.8s. Predicting...


TabPFN inference:   0%|          | 0/4 [00:00<?, ?estimator/s]

[Stage 1 (v3.5)] Done in 31.0s | Test ROC-AUC: 0.9869 | Test PR-AUC: 0.9735


In [2]:
# Stage 2 TabPFN Experiment (1,000 Train -> 500 Test, standalone & self-contained)
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

# Load .env cleanly without hardcoding secrets in the notebook
_env_file = Path(".env")
if _env_file.exists():
    for _line in _env_file.read_text().splitlines():
        _line = _line.strip()
        if _line and not _line.startswith("#") and "=" in _line:
            _k, _v = _line.split("=", 1)
            os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))

print(f"[Stage 2] TABPFN_TOKEN set in env: {bool(os.environ.get('TABPFN_TOKEN'))}", flush=True)

t0 = time.perf_counter()
df_tr = pd.read_csv(Path("processed") / "stage2_tfm_train.csv")
df_te = pd.read_csv(Path("processed") / "stage2_tfm_test.csv")

tr_sub, _ = train_test_split(df_tr, train_size=1000, stratify=df_tr["default"], random_state=42)
te_sub, _ = train_test_split(df_te, train_size=500, stratify=df_te["default"], random_state=42)

_drop_meta = {"grade", "issue_year", "default"}
_feats = [c for c in df_tr.columns if c not in _drop_meta]
_cat_cols = [c for c in _feats if df_tr[c].dtype == object or str(df_tr[c].dtype) == "string"]
_num_cols = [c for c in _feats if c not in _cat_cols]

tfm_prep = ColumnTransformer(
    [
        ("num", SimpleImputer(strategy="median"), _num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ]), _cat_cols),
    ],
    remainder="drop",
)
X_tfm_tr = tfm_prep.fit_transform(tr_sub[_feats])
X_tfm_te = tfm_prep.transform(te_sub[_feats])
print(f"[Stage 2] Data ready: train={X_tfm_tr.shape}, test={X_tfm_te.shape}", flush=True)

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

try:
    print(f"[Stage 2] Trying TabPFN v3.5 (n_estimators=4, device='auto')...", flush=True)
    clf = TabPFNClassifier(n_estimators=4, device="auto", show_progress_bar=True, random_state=42)
    clf.fit(X_tfm_tr, tr_sub["default"])
    ver_used = "v3.5"
except Exception as e:
    print(f"[Stage 2] v3.5 unavailable ({type(e).__name__}: {e}); falling back to ungated TabPFN v2...", flush=True)
    clf = TabPFNClassifier.create_default_for_version(
        ModelVersion.V2, n_estimators=4, device="auto", show_progress_bar=True, random_state=42
    )
    clf.fit(X_tfm_tr, tr_sub["default"])
    ver_used = "v2"

print(f"[Stage 2 ({ver_used})] Fit done in {time.perf_counter() - t0:.1f}s. Predicting...", flush=True)
p_tfm = clf.predict_proba(X_tfm_te)[:, 1]
Path("artifacts").mkdir(exist_ok=True)
np.savez_compressed("artifacts/stage2_tabpfn_preds.npz", y_true=te_sub["default"].to_numpy(), y_prob=p_tfm)
print(
    f"[Stage 2 ({ver_used})] Done in {time.perf_counter() - t0:.1f}s | "
    f"Test ROC-AUC: {roc_auc_score(te_sub['default'], p_tfm):.4f} | "
    f"Test PR-AUC: {average_precision_score(te_sub['default'], p_tfm):.4f}",
    flush=True,
)

[Stage 2] TABPFN_TOKEN set in env: True
[Stage 2] Data ready: train=(1000, 33), test=(500, 33)
[Stage 2] Trying TabPFN v3.5 (n_estimators=4, device='auto')...
[Stage 2 (v3.5)] Fit done in 0.3s. Predicting...


TabPFN inference:   0%|          | 0/4 [00:00<?, ?estimator/s]

[Stage 2 (v3.5)] Done in 1.8s | Test ROC-AUC: 0.6688 | Test PR-AUC: 0.2525
